Practica rnap

Nombre: Domenica Mora

Realizar:

Implementar el algoritmo del perceptrón original de Rosenblatt (con función escalón como activación) para aprender la compuerta AND. Mostrar la evolución de los pesos en cada época, verificar la convergencia y calcular la ecuación de la frontera de decisión final.

In [157]:
import numpy as np #librería

In [158]:
# funcion escalón
def escalon(x):
    return 1 if x >= 0 else 0

In [159]:
class Perceptron:
    def __init__(self, n_entradas, lr=0.1):
        self.w = np.zeros(n_entradas)
        self.b = 0
        self.lr = lr

    def forward(self, x):
        z = np.dot(self.w, x) + self.b
        return escalon(z)

    def backward(self, x, y):
        y_pred = self.forward(x)
        error = y - y_pred

        # actualización
        self.w = self.w + self.lr * error * x
        self.b = self.b + self.lr * error

        return error


In [160]:
# Datos de AND
X = np.array([[0,0], [0,1], [1,0], [1,1]])
y = np.array([0,0,0,1])

In [161]:
X

array([[0, 0],
       [0, 1],
       [1, 0],
       [1, 1]])

In [162]:
y

array([0, 0, 0, 1])

In [163]:
# Crear modelo
perceptron = Perceptron(n_entradas=2, lr=0.1)

In [164]:
# Entrenamiento
# Se muestran pesos en cada época para ver convergencia

for epoca in range(20):
    error_total = 0

    for i in range(len(X)):
        error = perceptron.backward(X[i], y[i])
        error_total += abs(error)

    # Mostrar resumen por época (aquí van los pesos)
    print("Epoca:", epoca,
          "Error:", error_total,
          "Pesos:", perceptron.w,
          "Bias:", perceptron.b)

    if error_total == 0:
        print("Convergió en la época", epoca)
        break

Epoca: 0 Error: 2 Pesos: [0.1 0.1] Bias: 0.0
Epoca: 1 Error: 3 Pesos: [0.2 0.1] Bias: -0.1
Epoca: 2 Error: 3 Pesos: [0.2 0.1] Bias: -0.20000000000000004
Epoca: 3 Error: 0 Pesos: [0.2 0.1] Bias: -0.20000000000000004
Convergió en la época 3


In [165]:
# Predicciones finales
for i in range(len(X)):
    pred = perceptron.forward(X[i])
    print(X[i], "->", pred)
    # Frontera de decisión
print("\nFrontera de decisión final:")
print(f"{perceptron.w[0]}*x1 + {perceptron.w[1]}*x2 + {perceptron.b} = 0")

[0 0] -> 0
[0 1] -> 0
[1 0] -> 0
[1 1] -> 1

Frontera de decisión final:
0.2*x1 + 0.1*x2 + -0.20000000000000004 = 0


Realizar:

Construir una red neuronal multicapa completa que clasifique las tres especies del dataset Iris. Incluir: normalización de datos, ReLU en capas ocultas, softmax en la salida, entrenamiento con cross-entropy y evaluación con accuracy. Y esta debe contener la función def predecir(self, X): return np.argmax(self.forward(X), axis=1)

In [166]:
# Importar librerías
import numpy as np
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [167]:
# Activaciones
def relu(x):
    return np.maximum(0, x)

def relu_derivada(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [168]:
def one_hot(y, n_clases):
    onehot = np.zeros((len(y), n_clases))
    onehot[np.arange(len(y)), y] = 1
    return onehot

In [169]:
# Red neuronal multicapa
class MLP:
    def __init__(self, n_capas, lr=0.01):
        # Arquitectura
        self.lr = lr
        self.W = []
        self.b = []

        for i in range(len(n_capas)-1):
            self.W.append(np.random.randn(n_capas[i], n_capas[i+1]) * 0.1)
            self.b.append(np.zeros((1, n_capas[i+1])))

    def forward(self, X):
        self.A = [X]
        self.Z = []
        entrada = X

        for i in range(len(self.W)-1):
            z = np.dot(entrada, self.W[i]) + self.b[i]
            a = relu(z)
            self.Z.append(z)
            self.A.append(a)
            entrada = a

        z = np.dot(entrada, self.W[-1]) + self.b[-1]
        a = softmax(z)

        self.Z.append(z)
        self.A.append(a)
        return a

    def backward(self, y):
        m = y.shape[0]
        delta = self.A[-1] - y  # cross-entropy

        for i in range(len(self.W)-1, -1, -1):
            dw = np.dot(self.A[i].T, delta) / m
            db = np.sum(delta, axis=0, keepdims=True) / m

            if i > 0:
                delta = np.dot(delta, self.W[i].T) * relu_derivada(self.Z[i-1])

            self.W[i] -= self.lr * dw
            self.b[i] -= self.lr * db

    def entrenamiento(self, X, y, epocas=1000):
        for ep in range(epocas):
            salida = self.forward(X)
            self.backward(y)

            if ep % 100 == 0:
                loss = -np.mean(y * np.log(salida + 1e-8))
                print("Epoca", ep, "Loss", loss)

    def predecir(self, X):
        return np.argmax(self.forward(X), axis=1)

In [170]:
    def entrenamiento(self, X, y, epocas=1000):
        for ep in range(epocas):
            salida = self.forward(X)
            self.backward(y)

            if ep % 100 == 0:
                loss = -np.mean(y * np.log(salida + 1e-8))
                print("Epoca", ep)
                print("Loss", loss)


In [171]:
# Dataset Iris
iris = datasets.load_iris()
X = iris.data
y = iris.target

In [172]:
# Normalización
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [173]:
y = one_hot(y, 3)

In [174]:
# Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [175]:
np.random.seed(42)
red = MLP([4,8,3], lr=0.01)

In [176]:
red.entrenamiento(X_train, y_train, epocas=1000)

Epoca 0 Loss 0.36874311682716043
Epoca 100 Loss 0.3630563651339042
Epoca 200 Loss 0.35543047684705037
Epoca 300 Loss 0.33647293001448636
Epoca 400 Loss 0.2948165310802088
Epoca 500 Loss 0.24076906422365224
Epoca 600 Loss 0.20083116091297035
Epoca 700 Loss 0.17569069634173798
Epoca 800 Loss 0.15841191806523994
Epoca 900 Loss 0.14511552978285072


In [177]:
# Evaluación
y_pred = red.predecir(X_test)
y_real = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred == y_real)
print("\nAccuracy:", accuracy)


Accuracy: 0.8666666666666667


In [178]:
# Predicciones
for i in range(len(y_pred)):
    print("Real:", y_real[i], "Predicho:", y_pred[i])

Real: 1 Predicho: 1
Real: 0 Predicho: 0
Real: 2 Predicho: 2
Real: 1 Predicho: 2
Real: 1 Predicho: 2
Real: 0 Predicho: 0
Real: 1 Predicho: 1
Real: 2 Predicho: 2
Real: 1 Predicho: 2
Real: 1 Predicho: 1
Real: 2 Predicho: 2
Real: 0 Predicho: 0
Real: 0 Predicho: 0
Real: 0 Predicho: 0
Real: 0 Predicho: 0
Real: 1 Predicho: 2
Real: 2 Predicho: 2
Real: 1 Predicho: 1
Real: 1 Predicho: 1
Real: 2 Predicho: 2
Real: 0 Predicho: 0
Real: 2 Predicho: 2
Real: 0 Predicho: 0
Real: 2 Predicho: 2
Real: 2 Predicho: 2
Real: 2 Predicho: 2
Real: 2 Predicho: 2
Real: 2 Predicho: 2
Real: 0 Predicho: 0
Real: 0 Predicho: 0
